# Case Study: Furiosa — A Mad Max Saga (2024)

**Last Updated**: April 2026  
**Data Source**: DuckDB Database (`data/db/movies.duckdb`)  
**Status**: ✅ Complete

## Overview

This notebook examines the financial and audience reception of *Furiosa: A Mad Max Saga* (2024) — a prequel/spinoff of *Mad Max: Fury Road* (2015). Despite strong reviews and fan appreciation, Furiosa is widely considered a commercial disappointment. We investigate **why** by looking at:

1. **The full Mad Max franchise** — budget escalation vs. returns over 45 years
2. **Fury Road vs. Furiosa head-to-head** — the sequel that spent more and earned less than half
3. **2024 box office context** — how Furiosa stacks up against its big-budget peers
4. **The spinoff problem** — does removing the franchise lead doom a film financially?

The core argument: Warner Bros. increased the budget by 12% ($150M → $168M) while targeting a narrower audience — a Mad Max movie *without* Mad Max. The mismatch between investment and realistic market size made this a predictable underperformer.

## Setup

In [18]:
import os
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Project root setup
from pyprojroot import here
os.chdir(here())

# Project utilities
from ayne.utils.query_utils import execute_custom_query, get_db_client

In [19]:
# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"${x:,.0f}" if abs(x) > 1000 else f"{x:.2f}")

# Color palette
COLORS = {
    "fury_road": "#E63946",   # Bold red
    "furiosa": "#F4A261",     # Amber/orange
    "franchise": "#264653",   # Dark teal
    "comparison": "#2A9D8F", # Teal
    "neutral": "#6C757D",    # Gray
    "profit": "#2D6A4F",     # Green
    "loss": "#D62828",       # Red
}

def fmt_millions(val):
    """Format a number as $XM or $X.XB."""
    if abs(val) >= 1e9:
        return f"${val / 1e9:.1f}B"
    return f"${val / 1e6:.0f}M"

## 1. The Mad Max Franchise — 45 Years of Escalating Ambition

The Mad Max franchise began as one of the most profitable films ever made relative to budget. The original 1979 film was made for **$350,000** and grossed **$100M** worldwide — a staggering ~286x return. Let's trace how the economics evolved.

In [20]:
# Load all Mad Max franchise movies from the database
mad_max = execute_custom_query("""
    SELECT
        m.title,
        t.release_date,
        t.budget,
        t.revenue,
        t.runtime,
        t.vote_average AS tmdb_rating,
        t.vote_count AS tmdb_votes,
        o.imdb_rating,
        o.imdb_votes,
        o.rotten_tomatoes_rating AS rt_score,
        o.metascore,
        o.director,
        o.actors,
        o.rated,
        o.awards,
        o.box_office AS us_box_office
    FROM movies m
    JOIN tmdb_movies t ON m.tmdb_id = t.tmdb_id
    LEFT JOIN omdb_movies o ON m.imdb_id = o.imdb_id
    WHERE m.title LIKE '%Mad Max%' OR m.title LIKE '%Furiosa%'
    ORDER BY t.release_date
""")

# Only Fury Road has full OMDB data in the DB. Supplement the other entries
# with well-known public data so the franchise table is complete.
known_data = {
    "Mad Max": {
        "imdb_rating": 6.8, "rt_score": 90, "metascore": 73,
        "director": "George Miller", "rated": "R",
        "actors": "Mel Gibson, Joanne Samuel, Hugh Keays-Byrne",
    },
    "Mad Max 2": {
        "imdb_rating": 7.6, "rt_score": 94, "metascore": 77,
        "director": "George Miller", "rated": "R",
        "actors": "Mel Gibson, Bruce Spence, Michael Preston",
    },
    "Mad Max Beyond Thunderdome": {
        "imdb_rating": 6.3, "rt_score": 80, "metascore": 71,
        "director": "George Miller, George Ogilvie", "rated": "PG-13",
        "actors": "Mel Gibson, Tina Turner, Bruce Spence",
    },
    "Furiosa: A Mad Max Saga": {
        "imdb_rating": 7.5, "rt_score": 90, "metascore": 79,
        "director": "George Miller", "rated": "R",
        "actors": "Anya Taylor-Joy, Chris Hemsworth, Tom Burke",
    },
}

for title, data in known_data.items():
    mask = mad_max["title"] == title
    for col, val in data.items():
        if mask.any() and (pd.isna(mad_max.loc[mask, col].values[0]) or mad_max.loc[mask, col].values[0] is None):
            mad_max.loc[mask, col] = val

mad_max["release_year"] = pd.to_datetime(mad_max["release_date"]).dt.year
mad_max["roi"] = (mad_max["revenue"] / mad_max["budget"]).round(2)
mad_max["profit"] = mad_max["revenue"] - mad_max["budget"]

display_cols = [
    "title", "release_year", "budget", "revenue", "profit", "roi",
    "tmdb_rating", "imdb_rating", "rt_score", "director", "rated",
]
mad_max[display_cols]

,title,release_year,budget,revenue,profit,roi,tmdb_rating,imdb_rating,rt_score,director,rated
0,Mad Max,1979,350000,100000000,99650000,285.71,6.68,6.80,90,George Miller,R
1,Mad Max 2,1981,2000000,24600832,22600832,12.30,7.40,7.60,94,George Miller,R
2,Mad Max Beyond Thunderdome,1985,10000000,36230219,26230219,3.62,6.21,6.30,80,"George Miller, George Ogilvie",PG-13
3,Mad Max: Fury Road,2015,150000000,378858340,228858340,2.53,7.63,8.10,97,George Miller,R
4,Furiosa: A Mad Max Saga,2024,168000000,174287546,6287546,1.04,7.46,7.50,90,George Miller,R


### Budget vs. Revenue Across the Franchise

The chart below illustrates the trajectory of the franchise. Note how budgets have exploded — particularly the jump from the original trilogy to the modern entries — while returns have not scaled proportionally.

In [21]:
# --- Chart 1: Budget vs Revenue (log scale to handle the $350K → $168M range) ---
fig = go.Figure()

fig.add_trace(go.Bar(
    x=mad_max["title"], y=mad_max["budget"],
    name="Budget", marker_color="#457B9D",
    text=mad_max["budget"].apply(fmt_millions), textposition="outside",
))
fig.add_trace(go.Bar(
    x=mad_max["title"], y=mad_max["revenue"],
    name="Revenue", marker_color="#E63946",
    text=mad_max["revenue"].apply(fmt_millions), textposition="outside",
))

fig.update_layout(
    title="Mad Max Franchise — Budget vs. Revenue",
    height=500, barmode="group",
    template="plotly_white",
    yaxis_title="USD ($) — Log Scale",
    yaxis_type="log",
    yaxis_range=[4.5, 9.5],  # 10^4.5 (~$30K) to 10^9.5 (~$3B)
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig.show()

In [22]:
# --- Chart 2: ROI (capped axis with annotation for the 286x outlier) ---
fig = go.Figure()

# Cap Mad Max 1979's ROI for display; annotate the real value
roi_capped = mad_max["roi"].clip(upper=20)

fig.add_trace(go.Bar(
    x=mad_max["title"],
    y=roi_capped,
    marker_color=[
        COLORS["profit"] if r > 2.5 else COLORS["furiosa"] if r > 1 else COLORS["loss"]
        for r in mad_max["roi"]
    ],
    text=[
        f"{r:.0f}x" if r > 20 else f"{r:.1f}x"
        for r in mad_max["roi"]
    ],
    textposition="outside",
))

# Break-even reference line
fig.add_hline(y=2.5, line_dash="dash", line_color="gray",
              annotation_text="~2.5x break-even (incl. marketing)")

# Annotate the clipped bar
max_roi_idx = mad_max["roi"].idxmax()
fig.add_annotation(
    x=mad_max.loc[max_roi_idx, "title"],
    y=20,
    text=f"Actual: {mad_max.loc[max_roi_idx, 'roi']:.0f}x (bar clipped)",
    showarrow=True, arrowhead=2, ay=-40,
    font=dict(size=11, color=COLORS["profit"]),
)

fig.update_layout(
    title="Mad Max Franchise - Return on Investment",
    yaxis_title="ROI Multiple (Revenue / Budget)",
    yaxis_range=[0, 25],
    template="plotly_white",
    height=450, showlegend=False,
)
fig.show()

### Key Takeaways — Franchise Trajectory

- **Mad Max (1979)** is one of the most profitable films ever made on an ROI basis — ~286x return on a \$350K budget. The log-scale budget chart shows just how micro the original investment was.
- The original trilogy had **modest budgets** and delivered solid returns. All three were directed by George Miller and all were critically well-received (80–94% on RT).
- **Fury Road (2015)** was a massive creative success (8.1 IMDb, 97% RT, 6 Oscars) but its ROI of ~2.5x was marginal for a \$150M blockbuster. Studios typically need ~2.5x gross revenue to break even after marketing and distribution costs.
- **Furiosa (2024)** repeated Fury Road's budget class (\$168M) but earned less than half the gross revenue. Despite strong reviews (7.5 IMDb, 90% RT), the ROI of ~1.04x means a clear financial loss.

> 💡 **The pattern**: George Miller makes brilliant films — every entry in this franchise is rated above 6.0 on IMDb — but the franchise economics have been on a downward trajectory as budgets inflated faster than the audience grew.

---

## 2. Head-to-Head: Fury Road vs. Furiosa

This is the core comparison. Same director, same franchise, same world — but dramatically different financial outcomes. Let's break down exactly where Furiosa underperformed.

In [23]:
# Side-by-side comparison table (now enriched with supplemental public data)
fury_road = mad_max[mad_max["title"].str.contains("Fury Road")].iloc[0]
furiosa = mad_max[mad_max["title"].str.contains("Furiosa")].iloc[0]

def safe_pct(val):
    return f"{int(val)}%" if pd.notna(val) else "N/A"

def safe_fmt(val, fmt=None):
    if pd.isna(val) or val is None:
        return "N/A"
    return fmt(val) if fmt else val

comparison = pd.DataFrame({
    "Metric": [
        "Release Year", "Budget", "Worldwide Gross", "Gross Profit (Revenue - Budget)",
        "ROI Multiple", "Runtime (min)", "Age Rating",
        "TMDB Rating", "IMDb Rating", "Rotten Tomatoes", "Metascore",
        "TMDB Vote Count", "Director", "Lead Cast",
    ],
    "Mad Max: Fury Road": [
        2015, fmt_millions(fury_road["budget"]), fmt_millions(fury_road["revenue"]),
        fmt_millions(fury_road["profit"]),
        f"{fury_road['roi']:.2f}x", int(fury_road["runtime"]),
        fury_road["rated"],
        fury_road["tmdb_rating"], fury_road["imdb_rating"],
        safe_pct(fury_road["rt_score"]), safe_fmt(fury_road.get("metascore"), lambda x: int(x)),
        f"{int(fury_road['tmdb_votes']):,}",
        fury_road["director"], fury_road["actors"],
    ],
    "Furiosa: A Mad Max Saga": [
        2024, fmt_millions(furiosa["budget"]), fmt_millions(furiosa["revenue"]),
        fmt_millions(furiosa["profit"]),
        f"{furiosa['roi']:.2f}x", int(furiosa["runtime"]),
        furiosa["rated"],
        furiosa["tmdb_rating"], furiosa["imdb_rating"],
        safe_pct(furiosa["rt_score"]), safe_fmt(furiosa.get("metascore"), lambda x: int(x)),
        f"{int(furiosa['tmdb_votes']):,}",
        furiosa["director"], furiosa["actors"],
    ],
})

comparison.style.set_properties(**{"text-align": "left"}).set_table_styles(
    [{"selector": "th", "props": [("text-align", "left")]}]
)

,Metric,Mad Max: Fury Road,Furiosa: A Mad Max Saga
0,Release Year,2015,2024
1,Budget,$150M,$168M
2,Worldwide Gross,$379M,$174M
3,Gross Profit (Revenue - Budget),$229M,$6M
4,ROI Multiple,2.53x,1.04x
5,Runtime (min),121,149
6,Age Rating,R,R
7,TMDB Rating,7.627000,7.460000
8,IMDb Rating,8.100000,7.500000
9,Rotten Tomatoes,97%,90%


In [24]:
# Visual comparison
metrics = ["Budget", "Revenue", "Profit"]
fury_vals = [fury_road["budget"], fury_road["revenue"], fury_road["profit"]]
furiosa_vals = [furiosa["budget"], furiosa["revenue"], furiosa["profit"]]

fig = go.Figure()
fig.add_trace(go.Bar(
    x=metrics, y=fury_vals, name="Fury Road (2015)",
    marker_color=COLORS["fury_road"],
    text=[fmt_millions(v) for v in fury_vals], textposition="outside",
))
fig.add_trace(go.Bar(
    x=metrics, y=furiosa_vals, name="Furiosa (2024)",
    marker_color=COLORS["furiosa"],
    text=[fmt_millions(v) for v in furiosa_vals], textposition="outside",
))

fig.update_layout(
    title="Fury Road vs. Furiosa — Financial Comparison",
    barmode="group", template="plotly_white",
    yaxis_title="USD ($)", height=450,
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig.show()

### The Numbers Tell the Story

- **Budget increased 12%**: \$150M → \$168M. Warner Bros. bet even bigger on the follow-up.
- **Revenue dropped 54%**: \$379M → \$174M. Furiosa earned less than half of what Fury Road did.
- **Critical reception held up**: Furiosa scored 90% on RT vs. Fury Road's 97%, and 7.5 vs 8.1 on IMDb — a strong showing by any standard. Both earned Metascores in the high 70s–90.
- **Fury Road was already marginal**: At ~2.5x ROI, Fury Road *barely* broke even once marketing costs (typically \$100M+ for tentpoles) are factored in.
- **Furiosa at 1.04x ROI**: The raw gross barely covered the production budget, let alone marketing. Industry estimates put the total loss at **\$100–150M**.

> ⚠️ **The general rule of thumb**: A film needs to earn roughly **2.5x its production budget** in worldwide gross to break even, because theaters keep ~50% of ticket sales and marketing typically adds another 50-100% of the production budget.

---

## 3. Context — 2024 Big-Budget Box Office

Was 2024 just a bad year for blockbusters, or was Furiosa's underperformance specific? Let's compare it against other films with budgets over \$100M released around the same period.

In [25]:
# Load 2024 big-budget movies for context
peers_2024 = execute_custom_query("""
    SELECT
        t.title,
        t.release_date,
        t.budget,
        t.revenue,
        t.vote_average AS tmdb_rating,
        t.genres,
        ROUND(CAST(t.revenue AS DOUBLE) / t.budget, 2) AS roi
    FROM tmdb_movies t
    WHERE t.budget >= 100000000
      AND t.revenue > 0
      AND t.release_date BETWEEN '2024-01-01' AND '2024-12-31'
    ORDER BY roi DESC
""")

peers_2024["is_furiosa"] = peers_2024["title"].str.contains("Furiosa")
print(f"Found {len(peers_2024)} big-budget movies from 2024")
peers_2024[["title", "budget", "revenue", "roi", "tmdb_rating"]]

Found 27 big-budget movies from 2024


,title,budget,revenue,roi,tmdb_rating
0,Despicable Me 4,100000000,969597394,9.70,7.01
1,Inside Out 2,200000000,1698863816,8.49,7.55
2,Moana 2,150000000,1059242164,7.06,7.03
3,Deadpool & Wolverine,200000000,1338073645,6.69,7.57
4,Wicked,150000000,758854096,5.06,6.91
5,Yolo,100000000,433606094,4.34,6.58
6,Bad Boys: Ride or Die,100000000,404547819,4.05,7.35
7,Sonic the Hedgehog 3,122000000,492162604,4.03,7.60
8,Venom: The Last Dance,120000000,478937618,3.99,6.68
9,Godzilla x Kong: The New Empire,150000000,571750016,3.81,7.08


In [26]:
# Scatter plot: Budget vs Revenue for 2024 big-budget films
fig = px.scatter(
    peers_2024,
    x="budget", y="revenue",
    size="tmdb_rating", size_max=20,
    color="is_furiosa",
    color_discrete_map={True: COLORS["furiosa"], False: COLORS["comparison"]},
    hover_name="title",
    hover_data={"budget": ":$,.0f", "revenue": ":$,.0f", "roi": ":.2f", "is_furiosa": False},
    labels={"budget": "Production Budget ($)", "revenue": "Worldwide Gross ($)"},
    title="2024 Big-Budget Films — Budget vs. Revenue (Furiosa highlighted)",
    template="plotly_white",
)

# Add break-even line (2.5x)
max_budget = peers_2024["budget"].max() * 1.1
fig.add_trace(go.Scatter(
    x=[0, max_budget], y=[0, max_budget * 2.5],
    mode="lines", line=dict(dash="dash", color="gray"),
    name="~2.5x break-even", showlegend=True,
))

# Annotate Furiosa
furiosa_row = peers_2024[peers_2024["is_furiosa"]].iloc[0]
fig.add_annotation(
    x=furiosa_row["budget"], y=furiosa_row["revenue"],
    text="Furiosa", showarrow=True, arrowhead=2,
    font=dict(size=12, color=COLORS["furiosa"]),
)

fig.update_layout(
    height=550, showlegend=True,
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig.show()

In [27]:
# ROI ranking bar chart for 2024
peers_sorted = peers_2024.sort_values("roi", ascending=True).copy()
peers_sorted["color"] = peers_sorted["is_furiosa"].map(
    {True: COLORS["furiosa"], False: COLORS["neutral"]}
)
# Highlight profitable vs unprofitable
peers_sorted.loc[
    (~peers_sorted["is_furiosa"]) & (peers_sorted["roi"] >= 2.5), "color"
] = COLORS["profit"]
peers_sorted.loc[
    (~peers_sorted["is_furiosa"]) & (peers_sorted["roi"] < 2.5), "color"
] = COLORS["loss"]

fig = go.Figure(go.Bar(
    y=peers_sorted["title"],
    x=peers_sorted["roi"],
    orientation="h",
    marker_color=peers_sorted["color"],
    text=peers_sorted["roi"].apply(lambda x: f"{x:.1f}x"),
    textposition="outside",
))

fig.add_vline(x=2.5, line_dash="dash", line_color="gray",
              annotation_text="~2.5x break-even")

fig.update_layout(
    title="2024 Big-Budget Films — ROI Ranking (Furiosa in orange)",
    xaxis_title="ROI (Revenue / Budget)",
    template="plotly_white",
    height=max(400, len(peers_sorted) * 30),
    margin=dict(l=250),
)
fig.show()

### 2024 Box Office Context

Furiosa landed near the **bottom of the pack** among 2024's big-budget releases. While some films with comparable budgets delivered massive returns (Inside Out 2 at ~8.5x, Deadpool & Wolverine at ~6.7x), Furiosa's 1.04x ROI puts it in the company of outright flops like *Joker: Folie à Deux* (1.09x) and *Argylle* (0.48x).

This wasn't a year where *every* big movie struggled — it was a year of stark **winners and losers**, and Furiosa was firmly in the latter camp.

---

## 4. The Spinoff Problem — Mad Max Without Mad Max

One theory for Furiosa's underperformance: audiences may be reluctant to follow a franchise when the titular character is removed. Let's see if this pattern holds more broadly by looking at other franchise spinoffs and their ROI compared to the "parent" film.

We'll compare a few notable cases from the database where a sequel or spinoff changed the lead character or premise.

In [28]:
# Notable franchise films vs. their spinoffs/reconfigurations
# We'll query specific titles from the database
spinoff_titles = [
    # Parent → Spinoff pairs
    "Mad Max: Fury Road",
    "Furiosa: A Mad Max Saga",
    "Joker",
    "Joker: Folie à Deux",
    "Venom",
    "Venom: The Last Dance",
    "Top Gun: Maverick",
    "Dune: Part Two",
    "Deadpool & Wolverine",
    "Spider-Man: No Way Home",
    "The Marvels",
    "Captain Marvel",
]

placeholders = ", ".join([f"'{t}'" for t in spinoff_titles])

spinoffs = execute_custom_query(f"""
    SELECT
        t.title,
        t.release_date,
        t.budget,
        t.revenue,
        t.vote_average AS tmdb_rating,
        ROUND(CAST(t.revenue AS DOUBLE) / t.budget, 2) AS roi
    FROM tmdb_movies t
    WHERE t.title IN ({placeholders})
      AND t.budget > 0 AND t.revenue > 0
    ORDER BY t.release_date
""")

spinoffs

,title,release_date,budget,revenue,tmdb_rating,roi
0,Mad Max: Fury Road,2015-05-13,150000000,378858340,7.63,2.53
1,Venom,2018-09-28,116000000,856085151,6.83,7.38
2,Captain Marvel,2019-03-06,152000000,1131416446,6.79,7.44
3,Joker,2019-10-01,55000000,1078958629,8.13,19.62
4,Spider-Man: No Way Home,2021-12-15,200000000,1921847111,7.94,9.61
5,Top Gun: Maverick,2022-05-21,170000000,1488732821,8.16,8.76
6,The Marvels,2023-11-08,274800000,206136825,5.96,0.75
7,Dune: Part Two,2024-02-27,190000000,714844358,8.10,3.76
8,Furiosa: A Mad Max Saga,2024-05-22,168000000,174287546,7.46,1.04
9,Deadpool & Wolverine,2024-07-24,200000000,1338073645,7.57,6.69


In [29]:
# Build franchise pairs for comparison
pairs = [
    ("Mad Max: Fury Road", "Furiosa: A Mad Max Saga", "Mad Max"),
    ("Joker", "Joker: Folie à Deux", "Joker"),
    ("Captain Marvel", "The Marvels", "Captain Marvel"),
    ("Venom", "Venom: The Last Dance", "Venom"),
]

pair_data = []
for parent_title, spinoff_title, franchise in pairs:
    parent = spinoffs[spinoffs["title"] == parent_title]
    child = spinoffs[spinoffs["title"] == spinoff_title]
    if not parent.empty and not child.empty:
        p = parent.iloc[0]
        c = child.iloc[0]
        pair_data.append({
            "franchise": franchise,
            "original": p["title"],
            "original_roi": p["roi"],
            "original_revenue": p["revenue"],
            "sequel": c["title"],
            "sequel_roi": c["roi"],
            "sequel_revenue": c["revenue"],
            "revenue_drop_pct": round((1 - c["revenue"] / p["revenue"]) * 100, 1),
            "roi_drop": round(p["roi"] - c["roi"], 2),
        })

pairs_df = pd.DataFrame(pair_data)
pairs_df[["franchise", "original", "original_roi", "sequel", "sequel_roi", "revenue_drop_pct"]]

,franchise,original,original_roi,sequel,sequel_roi,revenue_drop_pct
0,Mad Max,Mad Max: Fury Road,2.53,Furiosa: A Mad Max Saga,1.04,54.00
1,Joker,Joker,19.62,Joker: Folie à Deux,1.09,80.80
2,Captain Marvel,Captain Marvel,7.44,The Marvels,0.75,81.80
3,Venom,Venom,7.38,Venom: The Last Dance,3.99,44.10


In [30]:
# Grouped bar chart: Original vs Sequel ROI
fig = go.Figure()

fig.add_trace(go.Bar(
    x=pairs_df["franchise"], y=pairs_df["original_roi"],
    name="Original / Parent", marker_color=COLORS["profit"],
    text=pairs_df["original_roi"].apply(lambda x: f"{x:.1f}x"), textposition="outside",
))

fig.add_trace(go.Bar(
    x=pairs_df["franchise"], y=pairs_df["sequel_roi"],
    name="Sequel / Spinoff", marker_color=COLORS["loss"],
    text=pairs_df["sequel_roi"].apply(lambda x: f"{x:.1f}x"), textposition="outside",
))

fig.add_hline(y=2.5, line_dash="dash", line_color="gray",
              annotation_text="~2.5x break-even")

fig.update_layout(
    title="Franchise Sequels & Spinoffs — ROI Drop-Off",
    yaxis_title="ROI (Revenue / Budget)",
    barmode="group", template="plotly_white",
    height=450,
    legend=dict(orientation="h", yanchor="bottom", y=1.02),
)
fig.show()

In [31]:
# Revenue drop waterfall
fig = go.Figure(go.Waterfall(
    x=pairs_df["franchise"],
    y=-pairs_df["revenue_drop_pct"],
    text=pairs_df["revenue_drop_pct"].apply(lambda x: f"-{x:.0f}%"),
    textposition="outside",
    connector_line_color="rgba(0,0,0,0)",
    decreasing_marker_color=COLORS["loss"],
    increasing_marker_color=COLORS["loss"],
))

fig.update_layout(
    title="Revenue Drop: Original → Sequel/Spinoff (%)",
    yaxis_title="Revenue Change (%)",
    template="plotly_white",
    height=400,
    showlegend=False,
)
fig.show()

### The Spinoff Tax

The pattern is clear across multiple franchises:

- **Furiosa** dropped ~54% in revenue vs. Fury Road — a Mad Max movie without Mad Max.
- **Joker: Folie à Deux** dropped similarly — the sequel nobody asked for, with a bizarre musical pivot.
- **The Marvels** dropped vs. Captain Marvel — diluting the lead across three characters.

When studios change the formula (different lead, different tone, ensemble instead of solo), audiences often don't follow — especially at the same price point.

---

## 5. Budget Strategy — Where Did It Go Wrong?

Let's examine the budget decision in more detail. The central strategic error was **increasing the budget for a narrower audience**.

In [32]:
# What budget would have made Furiosa viable?
furiosa_revenue = furiosa["revenue"]
breakeven_budget = furiosa_revenue / 2.5  # Industry rule of thumb

budget_analysis = pd.DataFrame({
    "Scenario": [
        "Actual Budget",
        "Break-Even Budget (at $174M revenue)",
        "Fury Road's Budget (for reference)",
        "Budget Overshoot",
    ],
    "Amount": [
        furiosa["budget"],
        breakeven_budget,
        fury_road["budget"],
        furiosa["budget"] - breakeven_budget,
    ],
})

budget_analysis["Formatted"] = budget_analysis["Amount"].apply(fmt_millions)
budget_analysis

,Scenario,Amount,Formatted
0,Actual Budget,"$168,000,000",$168M
1,Break-Even Budget (at $174M revenue),"$69,715,018",$70M
2,Fury Road's Budget (for reference),"$150,000,000",$150M
3,Budget Overshoot,"$98,284,982",$98M


In [33]:
# What revenue did Furiosa need to break even at its actual budget?
needed_revenue = furiosa["budget"] * 2.5

fig = go.Figure()

categories = ["Budget", "Actual Revenue", "Revenue Needed\nfor Break-Even", "Revenue Gap"]
values = [
    furiosa["budget"],
    furiosa["revenue"],
    needed_revenue,
    needed_revenue - furiosa["revenue"],
]
colors = [COLORS["neutral"], COLORS["furiosa"], COLORS["franchise"], COLORS["loss"]]

fig.add_trace(go.Bar(
    x=categories, y=values,
    marker_color=colors,
    text=[fmt_millions(v) for v in values], textposition="outside",
))

fig.update_layout(
    title="Furiosa — The Revenue Gap",
    yaxis_title="USD ($)",
    template="plotly_white",
    height=450, showlegend=False,
)
fig.show()

### The Budget Miscalculation

- At \$174M in worldwide gross, Furiosa needed a production budget of roughly **\$70M** to break even (assuming typical marketing spend and theater revenue splits).
- Instead, it was made for **\$168M** — nearly 2.5x what the market could support.
- To break even at \$168M budget, it would have needed roughly **\$420M** in worldwide gross — more than Fury Road earned.
- The fundamental error: **budgeting for Fury Road's audience while making a film for a subset of that audience.**

> 💡 Fury Road had star power (Tom Hardy + Charlize Theron), the "Mad Max" name with the character present, 6 Oscar wins, and years of anticipation. Furiosa had a new lead (Anya Taylor-Joy + Chris Hemsworth), a prequel premise, and audience fatigue — yet cost *more* to make.

---

## 6. Audience Reception — Quality Wasn't the Problem

One important nuance: Furiosa wasn't a *bad* movie. It received strong critical and audience ratings. The failure was purely financial.

In [34]:
# TMDB ratings for all Mad Max franchise movies
fig = go.Figure()

fig.add_trace(go.Bar(
    x=mad_max["title"],
    y=mad_max["tmdb_rating"],
    marker_color=[
        COLORS["fury_road"] if "Fury Road" in t
        else COLORS["furiosa"] if "Furiosa" in t
        else COLORS["franchise"]
        for t in mad_max["title"]
    ],
    text=mad_max["tmdb_rating"].apply(lambda x: f"{x:.1f}"),
    textposition="outside",
))

fig.update_layout(
    title="Mad Max Franchise — TMDB Audience Ratings",
    yaxis_title="TMDB Rating",
    yaxis_range=[0, 10],
    template="plotly_white",
    height=400, showlegend=False,
)
fig.show()

In [35]:
# Rating vs ROI: Is quality correlated with returns in 2024?
fig = px.scatter(
    peers_2024,
    x="tmdb_rating", y="roi",
    size="budget", size_max=25,
    color="is_furiosa",
    color_discrete_map={True: COLORS["furiosa"], False: COLORS["comparison"]},
    hover_name="title",
    hover_data={"budget": ":$,.0f", "roi": ":.2f", "is_furiosa": False},
    labels={"tmdb_rating": "TMDB Rating", "roi": "ROI (Revenue / Budget)"},
    title="2024 Big-Budget Films — Quality vs. Financial Return",
    template="plotly_white",
)

fig.add_hline(y=2.5, line_dash="dash", line_color="gray",
              annotation_text="~2.5x break-even")

# Annotate Furiosa
fig.add_annotation(
    x=furiosa_row["tmdb_rating"], y=furiosa_row["roi"],
    text="Furiosa — good movie, bad investment",
    showarrow=True, arrowhead=2,
    font=dict(size=11, color=COLORS["furiosa"]),
)

fig.update_layout(height=500, showlegend=False)
fig.show()

### Quality ≠ Commercial Success

Furiosa sits in an interesting quadrant: **high quality, low return**. Its TMDB rating (7.46) places it in the top tier of 2024 releases — higher than several films that earned multiples of its gross. Its IMDb (7.5), Rotten Tomatoes (90%), and Metascore (79) all tell the same story: audiences and critics who saw it, liked it.

This underscores that the failure was not creative — it was a **business strategy failure**. The audience for a Furiosa origin story simply was not large enough to justify a \$168M production.

---

## Conclusions

### What Went Wrong With Furiosa?

1. **Budget mismatch**: WB priced Furiosa as a \$168M tentpole when its realistic ceiling was well below Fury Road's \$379M. Even Fury Road was only marginally profitable at that budget level.

2. **The spinoff tax**: Removing the titular character (Mad Max) from a Mad Max movie narrows the audience. This pattern repeated across multiple 2024 franchises — *Joker: Folie à Deux*, *The Marvels*, and others all saw steep drop-offs when they deviated from the original formula.

3. **Sequel fatigue in a crowded market**: 2024 demonstrated that audiences are increasingly selective. Films with strong brand recognition and faithful execution (*Inside Out 2*, *Deadpool & Wolverine*) thrived, while risky departures struggled.

4. **Quality wasn't the issue**: Furiosa was well-reviewed across the board — 7.5 IMDb, 90% RT, 79 Metascore, 7.46 TMDB. The problem was that not enough people chose to see it — a positioning and market-sizing problem, not a quality one.

### What Would Have Worked?

- **A \$70–80M budget** would have made Furiosa's \$174M gross comfortably profitable
- **Or**: the franchise needed Mad Max himself to justify the \$168M bet
- The original Mad Max trilogy proved that this franchise can deliver exceptional returns at **modest budgets** — George Miller doesn't need \$168M to tell a great story in this world

> 🎬 *Furiosa is a case study in how a great film can still be a financial disaster when the business strategy doesn't match the market reality.*